# Stock Movement Prediction Model Training

**Goal**: Predict future direction (up/down/flat) for US stocks over horizons {1, 5, 20} trading days using multi-source time-series data.

**Data Sources**:
- Yahoo Finance (yfinance): Primary OHLCV data
- Stooq: Fallback market data source
- FRED API: Macro economic indicators

**Model**: XGBoost multi-class classifier with walk-forward validation

**Critical**: This notebook implements strict data leakage prevention at every stage.

## 1. Imports and Configuration

In [ ]:
# Standard library
import os
import sys
import json
import pickle
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# Data handling
import pandas as pd
import numpy as np

# Data sources
import yfinance as yf
import requests
from fredapi import Fred

# ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Statistics
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All imports successful")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"XGBoost version: {xgb.__version__}")

In [ ]:
# Configuration
CONFIG = {
    # Paths
    "artifacts_dir": Path("../artifacts"),
    "data_dir": Path("../data"),
    "raw_data_dir": Path("../data/raw"),
    "processed_data_dir": Path("../data/processed"),
    
    # Data parameters
    "start_date": "2014-01-01",  # 10+ years of data
    "end_date": datetime.now().strftime("%Y-%m-%d"),
    
    # Prediction horizons (trading days)
    "horizons": [1, 5, 20],
    
    # Classification thresholds (returns)
    "up_threshold": 0.02,      # >2% return = UP
    "down_threshold": -0.02,   # <-2% return = DOWN
    # Between thresholds = FLAT
    
    # Model parameters
    "test_periods": 5,         # Number of walk-forward test periods
    "min_history_days": 500,   # Minimum history before first prediction
    
    # Feature engineering
    "rsi_period": 14,
    "macd_fast": 12,
    "macd_slow": 26,
    "macd_signal": 9,
    "bb_period": 20,
    "atr_period": 14,
    
    # Model version
    "model_version": f"v1.0_{datetime.now().strftime('%Y%m%d')}",
}

# Create directories
for dir_path in [CONFIG["artifacts_dir"], CONFIG["raw_data_dir"], CONFIG["processed_data_dir"]]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Check for FRED API key
try:
    FRED_API_KEY = os.environ["FRED_API_KEY"]
    print("✓ FRED_API_KEY found in environment")
except KeyError:
    print("❌ ERROR: FRED_API_KEY not found in environment variables")
    print("   Please set it: export FRED_API_KEY=your_api_key")
    FRED_API_KEY = None

print("\n" + "="*60)
print("Configuration:")
print(f"  Date range: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"  Horizons: {CONFIG['horizons']} days")
print(f"  Thresholds: UP>{CONFIG['up_threshold']:.1%}, DOWN<{CONFIG['down_threshold']:.1%}")
print(f"  Model version: {CONFIG['model_version']}")
print("="*60)

## 2. Data Ingestion

We'll implement functions to fetch data from multiple sources with proper error handling.

In [ ]:
def get_sp500_tickers(limit: Optional[int] = None) -> List[str]:
    """
    Fetch S&P 500 ticker list from Wikipedia.
    
    Args:
        limit: Optional limit on number of tickers (for testing)
        
    Returns:
        List of ticker symbols
    """
    try:
        url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
        tables = pd.read_html(url)
        sp500_table = tables[0]
        tickers = sp500_table['Symbol'].tolist()
        
        # Clean tickers (remove dots for compatibility)
        tickers = [t.replace('.', '-') for t in tickers]
        
        if limit:
            tickers = tickers[:limit]
            
        print(f"✓ Retrieved {len(tickers)} S&P 500 tickers")
        return tickers
    except Exception as e:
        print(f"❌ Error fetching S&P 500 tickers: {e}")
        # Fallback to manual list for testing
        fallback = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'JPM', 'V', 'JNJ']
        print(f"   Using fallback list of {len(fallback)} tickers")
        return fallback[:limit] if limit else fallback

# Get ticker list (limit for faster testing in skeleton mode)
TICKERS = get_sp500_tickers(limit=10)  # Remove limit for full training
print(f"Working with {len(TICKERS)} tickers: {TICKERS[:5]}...")